In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# %%bash 
# for f in drive/MyDrive/zipped_LOB_data/*; do
#     name=$(basename $f ".7z")
#     mkdir -p "data/extracted/$name"
#     7z x $f -o"data/extracted/$name" -y
# done


In [1]:
%load_ext autoreload
%autoreload 2


In [ ]:
from pathlib import Path 
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import torch 
import json
from datetime import datetime
from torch.utils.data import TensorDataset,DataLoader
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import copy
import warnings
warnings.filterwarnings("ignore")

DATA_ROOT = Path("data/extracted")
print(f'Root directory exists: {DATA_ROOT.exists()}')

symbol_dir_list = list(DATA_ROOT.glob("*"))

print(f'number of experiments: {len(symbol_dir_list)}')

from deeplob_library.train import train_model,test_model
from deeplob_library.models import DeepLOB
from deeplob_library.data import get_lists_of_tensors,get_train_val_test_sets,normalize_train_val_test,make_target_labels

BATCH_SIZE = 1024
for i,symbol_dir in enumerate(symbol_dir_list):
    orderbook_files = sorted(symbol_dir.glob("*orderbook_10.csv"))
    message_files = sorted(symbol_dir.glob("*message_10.csv"))
    #print directory folder name only
    print(f'experiment {i} on: {symbol_dir.name}')
        
    Xs,ys = get_lists_of_tensors(message_files,orderbook_files)

    X_train,y_train,X_val,y_val,X_test,y_test = get_train_val_test_sets(Xs,ys,val_days=1,test_days=1)

    X_train,X_val,X_test = normalize_train_val_test(X_train,X_val,X_test)
    y_train,y_val,y_test = make_target_labels(y_train,y_val,y_test)


    train_loader = DataLoader(TensorDataset(X_train,y_train),batch_size=BATCH_SIZE,shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val,y_val),batch_size=BATCH_SIZE,shuffle=False)
    test_loader = DataLoader(TensorDataset(X_test,y_test),batch_size=BATCH_SIZE,shuffle=False)


    CONFIG = {
        "LR":1e-3,
        "EPOCHS":10,
        "BATCH_SIZE":1024,
        "MODEL_NAME":"DeepLOB",
    }

    #VARIABLES
    LR = CONFIG["LR"]
    EPOCHS = CONFIG["EPOCHS"]
    BATCH_SIZE = CONFIG["BATCH_SIZE"]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = DeepLOB().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(),lr=LR)


    model,history = train_model(model,
                                train_loader,
                                val_loader,
                                criterion,
                                optimizer,
                                device,
                                EPOCHS)

    test_loss,test_acc = test_model(model,test_loader,criterion,device)

    print("--------------------------------")


Root directory exists: True
number of experiments: 3
experiment 0 on: AMD_2026-04-01_2026-04-10_10


In [39]:
import deeplob_library.data as data
print(data.get_train_val_test_sets)

AttributeError: module 'deeplob_library.data' has no attribute 'get_train_val_test_sets'